In [1]:
!nvidia-smi

Mon May  4 21:45:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:49:00.0 Off |                    0 |
| N/A   38C    P0             84W /  350W |     661MiB /  46068MiB |      0%   E. Process |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

%cd /projectnb/batmanlab/mragoza/lung-project/notebooks/bmc-4dct

/projectnb/batmanlab/mragoza/lung-project/notebooks/bmc-4dct


In [3]:
import sys, os, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import pydicom
import nibabel as nib

sys.path.append('/projectnb/batmanlab/mragoza/lung-project')
import project

torch.cuda.is_available()

True

In [4]:
%ls /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/extracted/2026-03-25_batch/4DCT

4D0010_1/  4D002_1/  4D002_3/  4D004_2/  4D007_1/  4D007_3/  4D008_2/
4D001_1/   4D002_2/  4D004_1/  4D006_1/  4D007_2/  4D008_1/  4D009_1/


In [5]:
data_root = Path('/restricted/projectnb/batmanlab/mragoza/BMC-4DCT/')

batch_groups = {
    '2026-03-25_batch': ['4DCT', 'Prior', 'Prior2', 'Prior3', 'Next1', 'Next2']
}

In [90]:
batch_name = '2026-03-25_batch'

def _parse_series_desc(s: str):
    import re
    if pd.isna(s):
        return
    if s.startswith('For Series:'):
        return
    if s.startswith('E1 For Series:'):
        return
    if s.startswith('Extended FOV'):
        return
    if s.startswith('Lung'):
        return
    if s.startswith('Thoracic'):
        return
    if s.startswith('Abdomen'):
        return
    if s.startswith('Exam Summary'):
        return

    match = re.search('(\d+%)', s)
    if match:
        return '4dct'

    raise RuntimeError(f'Failed to parse series description: {s!r}')

image_meta = []
for group_name in batch_groups[batch_name][:1]:
    group_root = data_root / 'converted' / batch_name / group_name

    for path in sorted(group_root.iterdir()):
        study_name = path.stem

        match = re.match(r'^4D00(\d+)_(\d+)$', study_name)
        if not match:
            print(f'WARNING: {study_name!r} does not match pattern', file=sys.stderr)
            continue

        subject_id = match.group(1)
        study_num = match.group(2)

        for fname in path.glob('*.json'):
            m = project.core.fileio.load_json(fname)
            try:
                modality = m.get('Modality', pd.NA)
                dev_maker = m.get('Manufacturer', pd.NA)
                dev_model = m.get('ManufacturersModelName', pd.NA)
                dev_serial = m.get('DeviceSerialNumber', pd.NA)
                series_num = m.get('SeriesNumber', pd.NA)
                study_desc = m.get('StudyDescription', pd.NA)
                series_desc = m.get('SeriesDescription', pd.NA)
                protoc_name = m.get('ProtocolName', pd.NA)
                image_type = ','.join(m['ImageType'])
                image_name = fname.stem
                _parse_series_desc(series_desc)

            except RuntimeError:
                project.core.utils.pprint(m, 2, 25)
                raise

            image_meta.append([
                batch_name,
                group_name,
                study_name,
                subject_id,
                study_num,
                series_num,
                modality,
                dev_maker,
                dev_model,
                dev_serial,
                study_desc,
                series_desc,
                protoc_name,
                image_type,
                image_name
            ])

image_meta = pd.DataFrame(image_meta, columns=[
    'batch_name',
    'group_name',
    'study_name',
    'subject_id',
    'study_number',
    'series_number',
    'modality',
    'device_maker',
    'device_model',
    'device_serial',
    'study_description',
    'series_description',
    'protocol_name',
    'image_type',
    'image_name'
])
image_meta

Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D0010_1/4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_399.json
Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D0010_1/4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_306.json
Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D0010_1/4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_302.json
Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D0010_1/4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_307.json
Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D0010_1/4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_304.json
Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D0010_1/4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_305.json
Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batc

,batch_name,group_name,study_name,subject_id,study_number,series_number,modality,device_maker,device_model,device_serial,study_description,series_description,protocol_name,image_type,image_name
0,2026-03-25_batch,4DCT,4D0010_1,10,1,399,NaN,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,For Series: 301 - 310,4D CT Chest /Onco Body,"DERIVED,SECONDARY,RESPIRATORY,WAVE",4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_399
1,2026-03-25_batch,4DCT,4D0010_1,10,1,306,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,50 50% Linear,4D CT Chest /Onco Body,"ORIGINAL,PRIMARY,AXIAL",4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_306
2,2026-03-25_batch,4DCT,4D0010_1,10,1,302,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,10 10% Linear,4D CT Chest /Onco Body,"ORIGINAL,PRIMARY,AXIAL",4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_302
3,2026-03-25_batch,4DCT,4D0010_1,10,1,307,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,60 60% Linear,4D CT Chest /Onco Body,"ORIGINAL,PRIMARY,AXIAL",4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_307
4,2026-03-25_batch,4DCT,4D0010_1,10,1,304,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,30 30% Linear,4D CT Chest /Onco Body,"ORIGINAL,PRIMARY,AXIAL",4D0010_1_4D_CT_Chest_Onco_Body_20250313135722_304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,2026-03-25_batch,4DCT,4D009_1,9,1,301,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,"Extended FOV, 10 10% E1 Linear",4D CT Chest /Onco Body,"DERIVED,SECONDARY,AXIAL",4D009_1_4D_CT_Chest_Onco_Body_20250225134258_301
132,2026-03-25_batch,4DCT,4D009_1,9,1,304,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,"Extended FOV, 40 40% E1 Linear",4D CT Chest /Onco Body,"DERIVED,SECONDARY,AXIAL",4D009_1_4D_CT_Chest_Onco_Body_20250225134258_304
133,2026-03-25_batch,4DCT,4D009_1,9,1,302,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,"Extended FOV, 20 20% E1 Linear",4D CT Chest /Onco Body,"DERIVED,SECONDARY,AXIAL",4D009_1_4D_CT_Chest_Onco_Body_20250225134258_302
134,2026-03-25_batch,4DCT,4D009_1,9,1,305,CT,Philips,Big Bore,20006,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,"Extended FOV, 50 50% E1 Linear",4D CT Chest /Onco Body,"DERIVED,SECONDARY,AXIAL",4D009_1_4D_CT_Chest_Onco_Body_20250225134258_305


In [91]:
image_meta[image_meta.group_name == '4DCT'].iloc[0]

batch_name                                             2026-03-25_batch
group_name                                                         4DCT
study_name                                                     4D0010_1
subject_id                                                           10
study_number                                                          1
series_number                                                       399
modality                                                            NaN
device_maker                                                    Philips
device_model                                                   Big Bore
device_serial                                                     20006
study_description     RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...
series_description                                For Series: 301 - 310
protocol_name                                    4D CT Chest /Onco Body
image_type                           DERIVED,SECONDARY,RESPIRATO

In [92]:
image_meta.study_description.unique()

<ArrowStringArray>
[            'RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY FIELDS',
                     'CT For Placement of Radiation Therapy Fields',
 'RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY FIELDS WITH CONTRA']
Length: 3, dtype: str

In [93]:
image_meta.protocol_name.unique()

<ArrowStringArray>
[                    '4D CT Chest /Onco Body',
         '4D Chest W/O-W CONTRAST /Onco Body',
                      'SBRT 4D CT /Onco Body',
                          '4D CHEST/OncoBody',
                'SBRT BODY /Onco CK/SBRT/SRS',
                                    'Unknown',
               '4D CT Chest /Onco CyberKnife',
                               'Exam Summary',
 'SBRT 4D CT LiverW/WO CON /Onco CK/SBRT/SRS']
Length: 9, dtype: str

In [147]:
def _parse_series_desc(s):
    import re
    if pd.isna(s):
        return 'n/a'
    m = re.search(r'(\d+%)', s)
    if not m:
        return 'n/a'
    return m.group(1)

def _parse_protocol_name(s):
    if 'Chest' in s:
        return 'lung'
    if 'Liver' in s:
        return 'liver'
    return 'unknown'

image_meta['resp_state'] = image_meta.series_description.map(_parse_series_desc)
image_meta['anat_region'] = image_meta.protocol_name.map(_parse_protocol_name)

In [148]:
show_cols = ['image_name', 'study_description', 'protocol_name', 'series_number', 'series_description', 'resp_state', 'anat_region']

image_meta[image_meta.subject_id == '1'][show_cols]

,image_name,study_description,protocol_name,series_number,series_description,resp_state,anat_region
12,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_308,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,308,70 70% Linear,70%,lung
13,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_302,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,302,10 10% Linear,10%,lung
14,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_305,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,305,40 40% Linear,40%,lung
15,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_306,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,306,50 50% Linear,50%,lung
16,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_310,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,310,90 90% Linear,90%,lung
17,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_307,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,307,60 60% Linear,60%,lung
18,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_301,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,301,0 0% Linear,0%,lung
19,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_303,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,303,20 20% Linear,20%,lung
20,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_201,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,201,Lung,n/a,lung
21,4D001_1_4D_CT_Chest_Onco_Body_20241231083317_309,RAD ONC CT FOR PLACEMENT OF RADIATION THERAPY ...,4D CT Chest /Onco Body,309,80 80% Linear,80%,lung


In [149]:
image_meta.to_csv(data_root / 'metadata' / 'images.csv', index=False)

In [150]:
image_meta.resp_state.unique()

<ArrowStringArray>
['n/a', '50%', '10%', '60%', '30%', '40%', '90%', '80%', '70%', '20%', '0%']
Length: 11, dtype: str

In [186]:
%autoreload
import project.datasets.bmc4dct

ds = project.datasets.bmc4dct.BMC4DCTDataset(data_root)
ds

In [187]:
ds.source_path(1, 1, ds.EE_RESP_STATE, asset_type='image').exists()

<string>:3: PerformanceWarning: indexing past lexsort depth may impact performance.


True

In [189]:
%autoreload
examples = ds.list_examples(subjects=[1], state_pairs=[(ds.EI_RESP_STATE, ds.EE_RESP_STATE)])
len(examples)

<string>:3: PerformanceWarning: indexing past lexsort depth may impact performance.


1

In [191]:
ex = examples[0]
project.core.utils.pprint(ex)

Example()
├── dataset:  'BMC-4DCT'
├── subject:  1
├── variant:  None
├── paths:    dict(len=3)
|   ├── 'ref_source':  PosixPath('/restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D001_1/4D001_1_4D_CT_Chest_Onco_Body_20241231083317_301.nii.gz')
|   ├── 'init_source': PosixPath('/restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D001_1/4D001_1_4D_CT_Chest_Onco_Body_20241231083317_301.nii.gz')
|   └── 'curr_source': PosixPath('/restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D001_1/4D001_1_4D_CT_Chest_Onco_Body_20241231083317_306.nii.gz')
└── metadata: dict(len=3)
    ├── 'study_num':  1
    ├── 'init_state': '0%'
    └── 'curr_state': '50%'


In [195]:
# source visualization
import project.core.transforms

nifti = project.core.fileio.load_nibabel(ex.paths['ref_source'])
array = nifti.get_fdata()
spacing = project.core.transforms.get_affine_spacing(nifti.affine)
array.shape, spacing

Loading /restricted/projectnb/batmanlab/mragoza/BMC-4DCT/converted/2026-03-25_batch/4DCT/4D001_1/4D001_1_4D_CT_Chest_Onco_Body_20241231083317_301.nii.gz


((512, 512, 175), array([0.9765625, 0.9765625, 2.       ]))

In [196]:
import project.visual.matplotlib as mpl_viz

mpl_viz.SliceViewer(array, spacing, cmap='Grays')

AttributeError: 'NoneType' object has no attribute 'refresh_all'